# 04 — Crop Suitability & Yield Prediction Models
**CamAgri Agricultural Risk Assessment Platform**

This notebook trains and evaluates the crop suitability classifier and yield regressor:
- **Suitability** — Given soil + climate conditions, rank all 14 crops by probability of success
  - Free tier: Decision Tree
  - Medium tier: Random Forest
  - Premium tier: Gradient Boosting
- **Yield prediction** — Estimate t/ha given farm-level inputs
  - Free tier: Linear Regression
  - Medium tier: Random Forest Regressor
  - Premium tier: Gradient Boosting Regressor

**Sections**
1. Setup & data loading
2. Suitability dataset construction
3. Suitability model training (3 tiers)
4. Suitability evaluation & SHAP
5. Yield model training (3 tiers)
6. Yield evaluation & SHAP
7. Cross-validation analysis
8. Save models

## 1. Setup & Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, cross_validate,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('[INFO] SHAP not installed: pip install shap')

warnings.filterwarnings('ignore')

DATA_ROOT  = Path('../data')
FEAT_DIR   = Path('../outputs/features')
MODELS_DIR = Path('../models')
OUTPUT_DIR = Path('../outputs/suitability_yield')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111',
    'axes.edgecolor':'#333333','text.color':'#cccccc',
    'axes.labelcolor':'#cccccc','axes.titlecolor':'#ffffff',
    'xtick.color':'#666666','ytick.color':'#666666',
    'grid.color':'#222222','grid.alpha':0.5,'font.size':11,
})
LIME    = '#CAFF00'
PALETTE = ['#CAFF00','#00CFFF','#FF6B6B','#FFB800','#A78BFA','#34D399','#FB923C','#F472B6',
           '#60A5FA','#4ADE80','#F87171','#FBBF24','#818CF8','#2DD4BF']

CROPS   = ['maize','rice','cassava','cocoyam','plantain','cocoa','coffee',
           'groundnut','beans','tomato','onion','potato','palm_oil','sorghum']
REGIONS = ['North West','South West','West','Littoral','Adamawa',
           'Far North','Centre','East','North','South']

print('Setup complete.')

In [ ]:
# Load datasets
try:
    yields_fe = pd.read_parquet(FEAT_DIR / 'yield_features.parquet')
    print('Loaded engineered yield features.')
except FileNotFoundError:
    yields_fe = pd.read_csv(DATA_ROOT / 'yields' / 'cameroon_yields.csv')
    print('Loaded raw yield data (run notebook 02 for full features).')

soil    = pd.read_csv(DATA_ROOT / 'soil'    / 'cameroon_soil.csv')
climate = pd.read_csv(DATA_ROOT / 'climate' / 'cameroon_climate.csv')

print(f'yields_fe : {yields_fe.shape}')
print(f'soil      : {soil.shape}')
print(f'climate   : {climate.shape}')

## 2. Suitability Dataset Construction

In [ ]:
# ── 2.1 Define suitability labels ─────────────────────────────────────────
# For each observation (location + conditions), the crop grown IS the suitable crop.
# We create a multi-class classification target: predict which crop is best.
#
# Additionally we create a suitability score per crop per condition vector
# using the GCI-based scoring from notebook 02.

SUIT_FEATURES = [
    'soil_ph','rainfall_mm','temperature_c','humidity_pct','elevation_m',
]
# Add engineered features if available
EXTRA_SUIT = [
    'soil_ph_deviation','rain_deficit','rain_surplus','temp_deviation',
    'gci','rain_optimal_band','soil_ph_optimal','heat_stress','cold_stress',
]
SUIT_FEATURES += [c for c in EXTRA_SUIT if c in yields_fe.columns]

print(f'Suitability features ({len(SUIT_FEATURES)}): {SUIT_FEATURES}')

In [ ]:
# ── 2.2 Build classification dataset ──────────────────────────────────────
le_crop = LabelEncoder().fit(CROPS)
joblib.dump(le_crop, MODELS_DIR / 'le_crop.pkl')

suit_df = yields_fe[SUIT_FEATURES + ['crop']].dropna().copy()
suit_df['crop_label'] = le_crop.transform(suit_df['crop'])

X_suit = suit_df[SUIT_FEATURES].values
y_suit = suit_df['crop_label'].values

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_suit, y_suit, test_size=0.20, stratify=y_suit, random_state=42
)

scaler_suit = RobustScaler()
X_tr_s_sc = scaler_suit.fit_transform(X_tr_s)
X_te_s_sc = scaler_suit.transform(X_te_s)
joblib.dump(scaler_suit, MODELS_DIR / 'suitability_scaler.pkl')

print(f'Train: {X_tr_s.shape}  |  Test: {X_te_s.shape}')
print(f'Classes: {len(le_crop.classes_)}')

In [ ]:
# ── 2.3 Class balance ─────────────────────────────────────────────────────
class_counts = pd.Series(y_suit).map(dict(enumerate(le_crop.classes_))).value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(class_counts.index, class_counts.values,
              color=PALETTE[:len(class_counts)], edgecolor='none')
ax.set_title('Suitability Dataset — Class Balance (samples per crop)')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=35)
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'suit_class_balance.png', dpi=140, bbox_inches='tight')
plt.show()

## 3. Suitability Model Training (3 Tiers)

In [ ]:
# ── 3.1 Define tier models ────────────────────────────────────────────────
suit_models = {
    'free':    DecisionTreeClassifier(
                   max_depth=8, min_samples_split=10,
                   class_weight='balanced', random_state=42),
    'medium':  RandomForestClassifier(
                   n_estimators=200, max_depth=14, min_samples_split=5,
                   class_weight='balanced', random_state=42, n_jobs=-1),
    'premium': GradientBoostingClassifier(
                   n_estimators=300, learning_rate=0.05, max_depth=6,
                   subsample=0.8, random_state=42),
}

suit_results = {}

for tier, model in suit_models.items():
    print(f'\nTraining {tier} suitability model...')
    model.fit(X_tr_s_sc, y_tr_s)
    y_pred = model.predict(X_te_s_sc)

    acc    = accuracy_score(y_te_s, y_pred)
    f1_mac = f1_score(y_te_s, y_pred, average='macro',    zero_division=0)
    f1_wtd = f1_score(y_te_s, y_pred, average='weighted', zero_division=0)

    suit_results[tier] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': round(acc, 4),
        'f1_macro': round(f1_mac, 4),
        'f1_weighted': round(f1_wtd, 4),
    }
    print(f'  [{tier:8s}]  Accuracy={acc:.4f}  F1-macro={f1_mac:.4f}  F1-weighted={f1_wtd:.4f}')

In [ ]:
# ── 3.2 Confusion matrix — best model (premium) ───────────────────────────
best_tier  = 'premium'
cm = confusion_matrix(y_te_s, suit_results[best_tier]['y_pred'], normalize='true')

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(cm, ax=ax, annot=True, fmt='.2f', cmap='YlGn',
            xticklabels=le_crop.classes_, yticklabels=le_crop.classes_,
            linewidths=0.3, linecolor='#1a1a1a', annot_kws={'size': 8})
ax.set_title(f'Suitability Confusion Matrix — {best_tier.capitalize()} Tier (normalised)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'suit_confusion_matrix.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.3 Per-crop F1 scores across tiers ───────────────────────────────────
per_crop_f1 = {}
for tier, res in suit_results.items():
    rep = classification_report(y_te_s, res['y_pred'],
                                 target_names=le_crop.classes_, output_dict=True, zero_division=0)
    per_crop_f1[tier] = {crop: rep[crop]['f1-score'] for crop in le_crop.classes_}

f1_df = pd.DataFrame(per_crop_f1)

fig, ax = plt.subplots(figsize=(13, 7))
x = np.arange(len(f1_df))
w = 0.26
tier_colors = ['#444444', '#FFB800', LIME]
for i, (tier, c) in enumerate(zip(['free','medium','premium'], tier_colors)):
    ax.bar(x + (i-1)*w, f1_df[tier].values, w,
           label=tier.capitalize(), color=c, alpha=0.8, edgecolor='none')
ax.set_xticks(x); ax.set_xticklabels(f1_df.index, rotation=38, ha='right')
ax.set_ylabel('F1 Score'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Crop F1 Score by Tier — Suitability Model')
ax.legend(fontsize=10)
ax.axhline(0.7, color='#555555', lw=1, linestyle='--', label='0.7 threshold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'suit_per_crop_f1.png', dpi=140, bbox_inches='tight')
plt.show()

## 4. Suitability Evaluation & SHAP

In [ ]:
# ── 4.1 Feature importance (RF — medium tier) ─────────────────────────────
rf_suit = suit_results['medium']['model']
fi_suit = pd.Series(rf_suit.feature_importances_, index=SUIT_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(fi_suit)*0.42)))
colors  = [LIME if v > fi_suit.median() else '#444444' for v in fi_suit.values]
ax.barh(fi_suit.index, fi_suit.values, color=colors, edgecolor='none', height=0.65)
ax.set_title('Random Forest Feature Importance — Suitability Model')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'suit_feature_importance.png', dpi=140, bbox_inches='tight')
plt.show()

print('Top 5 suitability features:')
print(fi_suit.tail(5))

In [ ]:
# ── 4.2 SHAP values — premium GBM model ──────────────────────────────────
if SHAP_AVAILABLE:
    gbm_suit = suit_results['premium']['model']
    # Use a sample for speed
    X_shap_s = X_te_s_sc[:200]

    explainer_s  = shap.TreeExplainer(gbm_suit)
    shap_vals_s  = explainer_s.shap_values(X_shap_s)   # shape: [n_classes, n_samples, n_features]

    # Mean absolute SHAP across all classes
    if isinstance(shap_vals_s, list):
        mean_shap = np.mean([np.abs(sv) for sv in shap_vals_s], axis=0)
    else:
        mean_shap = np.abs(shap_vals_s)

    shap_mean_feat = pd.Series(mean_shap.mean(axis=0), index=SUIT_FEATURES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(4, len(shap_mean_feat)*0.42)))
    colors = [LIME if v > shap_mean_feat.median() else '#444444' for v in shap_mean_feat.values]
    ax.barh(shap_mean_feat.index, shap_mean_feat.values, color=colors, edgecolor='none', height=0.65)
    ax.set_title('Mean |SHAP| — Suitability Model (Premium GBM)')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'suit_shap.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('[SHAP not available] — install with: pip install shap')

In [ ]:
# ── 4.3 Suitability score inference example ───────────────────────────────
# Given a single field description, return a ranked suitability list
def predict_suitability(conditions: dict, tier='premium') -> pd.Series:
    """
    conditions: dict with keys matching SUIT_FEATURES
    Returns a Series of suitability scores per crop, sorted descending.
    """
    vec  = np.array([[conditions.get(f, 0) for f in SUIT_FEATURES]])
    vec_sc = scaler_suit.transform(vec)
    model  = suit_results[tier]['model']
    probas = model.predict_proba(vec_sc)[0]
    scores = pd.Series(probas, index=le_crop.classes_).sort_values(ascending=False)
    return scores

# North West region example
example_nw = {
    'soil_ph': 6.1, 'rainfall_mm': 1800, 'temperature_c': 20,
    'humidity_pct': 78, 'elevation_m': 1500,
    'soil_ph_deviation': abs(6.1-6.2), 'rain_deficit': 0,
    'rain_surplus': 0, 'temp_deviation': abs(20-25),
    'gci': 0.72, 'rain_optimal_band': 1,
    'soil_ph_optimal': 1, 'heat_stress': 0, 'cold_stress': 0,
}
scores_nw = predict_suitability(example_nw, tier='premium')
print('=== Suitability scores — North West (premium) ===')
print(scores_nw.round(4).to_string())

## 5. Yield Model Training (3 Tiers)

In [ ]:
# ── 5.1 Yield feature matrix ──────────────────────────────────────────────
YIELD_NUMERIC = [
    'soil_ph','rainfall_mm','temperature_c','humidity_pct','elevation_m',
    'land_size_ha','irrigation','fert_code',
]
YIELD_EXTRA = [
    'soil_ph_deviation','rain_deficit','rain_surplus','temp_deviation',
    'gci','input_intensity','irr_x_rain','fert_x_rain','gci_x_input',
]
YIELD_CAT   = ['crop_encoded','region_encoded']

# Build feature list from available columns
if 'fert_code' not in yields_fe.columns:
    fert_map = {'none':0,'organic':1,'inorganic':2,'mixed':3}
    yields_fe['fert_code'] = yields_fe['fertilizer_type'].map(fert_map).fillna(0)

if 'crop_encoded' not in yields_fe.columns:
    le_crop_loaded = LabelEncoder().fit(CROPS)
    yields_fe['crop_encoded']   = le_crop_loaded.transform(yields_fe['crop'])
    le_reg = LabelEncoder().fit(REGIONS)
    yields_fe['region_encoded'] = le_reg.transform(yields_fe['region'])

all_yield_feats = (YIELD_NUMERIC +
                   [c for c in YIELD_EXTRA if c in yields_fe.columns] +
                   [c for c in YIELD_CAT   if c in yields_fe.columns])

yield_df = yields_fe[all_yield_feats + ['yield_tons_per_ha']].dropna()
X_yield  = yield_df[all_yield_feats].values
y_yield  = yield_df['yield_tons_per_ha'].values

X_tr_y, X_te_y, y_tr_y, y_te_y = train_test_split(
    X_yield, y_yield, test_size=0.20, random_state=42
)

scaler_yield = RobustScaler()
X_tr_y_sc = scaler_yield.fit_transform(X_tr_y)
X_te_y_sc = scaler_yield.transform(X_te_y)
joblib.dump(scaler_yield, MODELS_DIR / 'yield_scaler.pkl')

print(f'Yield feature count: {len(all_yield_feats)}')
print(f'Train: {X_tr_y.shape}  |  Test: {X_te_y.shape}')

In [ ]:
# ── 5.2 Train tier models ─────────────────────────────────────────────────
yield_models = {
    'free':    Ridge(alpha=1.0),
    'medium':  RandomForestRegressor(
                   n_estimators=200, max_depth=14, min_samples_split=5,
                   random_state=42, n_jobs=-1),
    'premium': GradientBoostingRegressor(
                   n_estimators=300, learning_rate=0.05, max_depth=6,
                   subsample=0.8, random_state=42),
}

def yield_metrics(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    print(f'{label:40s}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'rmse':round(rmse,4),'mae':round(mae,4),'r2':round(r2,4),'mape':round(mape,2)}

yield_results = {}
print('=== YIELD MODEL TRAINING ===')
for tier, model in yield_models.items():
    model.fit(X_tr_y_sc, y_tr_y)
    y_pred = model.predict(X_te_y_sc)
    m = yield_metrics(y_te_y, y_pred, f'[{tier:8s}] {type(model).__name__}')
    yield_results[tier] = {'model': model, 'y_pred': y_pred, **m}

## 6. Yield Evaluation & SHAP

In [ ]:
# ── 6.1 Predicted vs actual scatter ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
tier_colors = ['#444444', '#FFB800', LIME]

for ax, (tier, res), c in zip(axes, yield_results.items(), tier_colors):
    y_pred = res['y_pred']
    ax.scatter(y_te_y, y_pred, alpha=0.35, s=12, color=c, edgecolors='none')
    lims = [min(y_te_y.min(), y_pred.min())*0.95,
            max(y_te_y.max(), y_pred.max())*1.05]
    ax.plot(lims, lims, 'white', lw=1.5, linestyle='--', label='Perfect')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f'{tier.capitalize()} — R²={res["r2"]:.4f}  MAPE={res["mape"]:.2f}%')
    ax.set_xlabel('Actual (t/ha)'); ax.set_ylabel('Predicted (t/ha)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

fig.suptitle('Yield Prediction: Actual vs Predicted — by Tier', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'yield_actual_vs_predicted.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.2 Residual analysis ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for i, (tier, res) in enumerate(yield_results.items()):
    resid = y_te_y - res['y_pred']
    # Residual vs fitted
    axes[0, i].scatter(res['y_pred'], resid, alpha=0.3, s=10,
                       color=tier_colors[i], edgecolors='none')
    axes[0, i].axhline(0, color='white', lw=1, linestyle='--')
    axes[0, i].set_title(f'{tier.capitalize()} — Residuals vs Fitted')
    axes[0, i].set_xlabel('Fitted'); axes[0, i].set_ylabel('Residual')
    # Residual histogram
    axes[1, i].hist(resid, bins=40, color=tier_colors[i], alpha=0.72, edgecolor='none', density=True)
    axes[1, i].set_title(f'{tier.capitalize()} — Residual Distribution')
    axes[1, i].set_xlabel('Residual (t/ha)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'yield_residuals.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 Per-crop yield prediction error ───────────────────────────────────
test_df = pd.DataFrame(X_te_y, columns=all_yield_feats)
test_df['actual']    = y_te_y
test_df['pred_prem'] = yield_results['premium']['y_pred']

# Map crop_encoded back to name
if 'crop_encoded' in test_df.columns:
    test_df['crop_name'] = le_crop.inverse_transform(
        test_df['crop_encoded'].round().astype(int).clip(0, len(CROPS)-1)
    )
    per_crop_mape = (
        test_df.groupby('crop_name')
        .apply(lambda g: np.mean(np.abs((g.actual - g.pred_prem)/(g.actual+1e-9)))*100)
        .sort_values(ascending=True)
        .rename('MAPE (%)')
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    colors  = [LIME if v < 10 else '#FFB800' if v < 20 else '#FF6B6B' for v in per_crop_mape.values]
    ax.barh(per_crop_mape.index, per_crop_mape.values, color=colors, edgecolor='none', height=0.65)
    ax.axvline(10, color='#555555', lw=1, linestyle='--', label='10% threshold')
    ax.set_title('Per-Crop MAPE — Premium Yield Model')
    ax.set_xlabel('MAPE (%)'); ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'yield_per_crop_mape.png', dpi=140, bbox_inches='tight')
    plt.show()
    print(per_crop_mape.round(2))

In [ ]:
# ── 6.4 SHAP values — yield premium model ────────────────────────────────
if SHAP_AVAILABLE:
    gbm_yield = yield_results['premium']['model']
    X_shap_y  = X_te_y_sc[:300]

    explainer_y = shap.TreeExplainer(gbm_yield)
    shap_vals_y = explainer_y.shap_values(X_shap_y)  # shape: [n_samples, n_features]

    # Summary beeswarm plot
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_vals_y, X_shap_y,
                      feature_names=all_yield_feats,
                      plot_type='bar', show=False,
                      color=LIME)
    plt.title('SHAP Feature Importance — Yield Model (Premium GBM)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'yield_shap_bar.png', dpi=140, bbox_inches='tight')
    plt.show()

    # Beeswarm
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_vals_y, X_shap_y,
                      feature_names=all_yield_feats, show=False)
    plt.title('SHAP Beeswarm — Yield Model')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'yield_shap_beeswarm.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    # Fallback: RF feature importance
    rf_yield = yield_results['medium']['model']
    fi_yield = pd.Series(rf_yield.feature_importances_, index=all_yield_feats).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(4, len(fi_yield)*0.38)))
    colors  = [LIME if v > fi_yield.median() else '#444444' for v in fi_yield.values]
    ax.barh(fi_yield.index, fi_yield.values, color=colors, edgecolor='none', height=0.65)
    ax.set_title('RF Feature Importance — Yield Model (Medium tier)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'yield_feature_importance.png', dpi=140, bbox_inches='tight')
    plt.show()

## 7. Cross-Validation Analysis

In [ ]:
# ── 7.1 5-fold CV for all yield models ───────────────────────────────────
print('=== 5-FOLD CROSS-VALIDATION — YIELD MODELS ===')
cv_results = {}

for tier, model in yield_models.items():
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r = cross_validate(
        model, X_tr_y_sc, y_tr_y, cv=cv,
        scoring=['r2','neg_root_mean_squared_error','neg_mean_absolute_error'],
        return_train_score=True, n_jobs=-1,
    )
    cv_results[tier] = cv_r
    r2   = cv_r['test_r2'].mean()
    rmse = -cv_r['test_neg_root_mean_squared_error'].mean()
    print(f'  [{tier:8s}]  R²={r2:.4f} ± {cv_r["test_r2"].std():.4f}'
          f'  RMSE={rmse:.4f} ± {-cv_r["test_neg_root_mean_squared_error"].std():.4f}')

In [ ]:
# ── 7.2 Learning curves ───────────────────────────────────────────────────
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
train_sizes_pct = np.linspace(0.1, 1.0, 8)

for ax, (tier, model), c in zip(axes, yield_models.items(), tier_colors):
    train_sizes, train_sc, val_sc = learning_curve(
        model, X_tr_y_sc, y_tr_y,
        train_sizes=train_sizes_pct, cv=4,
        scoring='r2', n_jobs=-1,
    )
    tr_mean, tr_std  = train_sc.mean(1), train_sc.std(1)
    val_mean, val_std= val_sc.mean(1),   val_sc.std(1)

    ax.plot(train_sizes, tr_mean,  color=c,       lw=2, label='Train R²')
    ax.plot(train_sizes, val_mean, color='white',  lw=2, linestyle='--', label='Val R²')
    ax.fill_between(train_sizes, tr_mean-tr_std,   tr_mean+tr_std,   alpha=0.15, color=c)
    ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1,  color='white')
    ax.set_title(f'{tier.capitalize()} — Learning Curve')
    ax.set_xlabel('Training samples'); ax.set_ylabel('R²')
    ax.legend(fontsize=9); ax.set_ylim(-0.1, 1.05); ax.grid(True, alpha=0.2)

fig.suptitle('Yield Model Learning Curves by Tier', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'yield_learning_curves.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 Stratified CV for suitability ─────────────────────────────────────
print('=== STRATIFIED 5-FOLD CV — SUITABILITY MODELS ===')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tier, model in suit_models.items():
    cv_acc = cross_val_score(model, X_tr_s_sc, y_tr_s, cv=skf,
                              scoring='accuracy', n_jobs=-1)
    cv_f1  = cross_val_score(model, X_tr_s_sc, y_tr_s, cv=skf,
                              scoring='f1_macro', n_jobs=-1)
    print(f'  [{tier:8s}]  Accuracy={cv_acc.mean():.4f} ± {cv_acc.std():.4f}'
          f'  F1-macro={cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

## 8. Save Models

In [ ]:
# ── 8.1 Save suitability models ───────────────────────────────────────────
for tier, res in suit_results.items():
    path = MODELS_DIR / f'suitability_{tier}.pkl'
    joblib.dump(res['model'], path)
    print(f'Saved: {path.name}')

# ── 8.2 Save yield models ─────────────────────────────────────────────────
for tier, res in yield_results.items():
    path = MODELS_DIR / f'yield_{tier}.pkl'
    joblib.dump(res['model'], path)
    print(f'Saved: {path.name}')

# ── 8.3 Save scalers & encoders ───────────────────────────────────────────
joblib.dump(scaler_suit,  MODELS_DIR / 'suitability_scaler.pkl')
joblib.dump(scaler_yield, MODELS_DIR / 'yield_scaler.pkl')
joblib.dump(le_crop,      MODELS_DIR / 'le_crop.pkl')

# ── 8.4 Save feature column lists ─────────────────────────────────────────
try:
    feat_cols = joblib.load(MODELS_DIR / 'feature_cols.pkl')
except FileNotFoundError:
    feat_cols = {}

feat_cols.update({
    'suitability': SUIT_FEATURES,
    'yield':       all_yield_feats,
})
joblib.dump(feat_cols, MODELS_DIR / 'feature_cols.pkl')
print('Feature column lists updated.')

print('\n=== ALL MODELS SAVED ===')

In [ ]:
# ── 8.5 Summary ───────────────────────────────────────────────────────────
print('=' * 65)
print('  CROP SUITABILITY & YIELD — NOTEBOOK SUMMARY')
print('=' * 65)
print()
print('  SUITABILITY MODELS:')
for tier, res in suit_results.items():
    print(f'    [{tier:8s}]  Accuracy={res["accuracy"]:.4f}  '
          f'F1-macro={res["f1_macro"]:.4f}  F1-weighted={res["f1_weighted"]:.4f}')

print()
print('  YIELD MODELS:')
for tier, res in yield_results.items():
    print(f'    [{tier:8s}]  R²={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}  MAPE={res["mape"]:.2f}%')

print()
print('  Saved artifacts:')
for f in sorted(MODELS_DIR.glob('*.pkl')):
    print(f'    {f.name:40s}  {f.stat().st_size/1024:.1f} KB')
print('=' * 65)